# Strict-causal multi-horizon Transformer

과거 방향 Conv1d와 causal attention으로 1·2·4·8 상대 시점 뒤 센서값의 평균과 불확실성을 예측한다. 정상 lot만으로 Gaussian NLL을 학습하고, 정상 reference의 경보 기준과 공개 holdout에서의 검토 순위를 평가한다.

이 노트북의 학습 cell은 checkpoint와 결과를 새로 저장한다. 입력은 전체 Step 1을 64개 상대 시점으로 보간한 trace다.


In [ ]:
from pathlib import Path  # 현재 위치에서 프로젝트와 공식 데이터 경로를 찾습니다.
import os  # 노트북 작업 폴더를 프로젝트 root로 바꿉니다.
import torch  # PyTorch CUDA 장치와 모델을 확인합니다.
project_dir = Path.cwd()  # Jupyter를 시작한 폴더를 기준점으로 잡습니다.
while not (project_dir / 'st_awfd_causal_gpt_multihorizon.py').exists() and project_dir != project_dir.parent:  # 새 스크립트가 나올 때까지 상위 폴더를 찾습니다.
    project_dir = project_dir.parent  # 현재 폴더에 없으면 부모 폴더를 확인합니다.
if not (project_dir / 'st_awfd_causal_gpt_multihorizon.py').exists():  # 프로젝트 폴더 밖에서 학습하는 일을 막습니다.
    raise FileNotFoundError('Open this notebook inside the pe_yield_lab project.')  # 필요한 실행 위치를 알립니다.
os.chdir(project_dir)  # 프로젝트 기준 data/models/results 경로를 활성화합니다.
print('Project:', project_dir)  # 실제 데이터가 있는 위치를 눈으로 확인합니다.
print('PyTorch:', torch.__version__)  # 재현 환경의 PyTorch 버전을 확인합니다.
print('CUDA available:', torch.cuda.is_available())  # GPU 사용 가능 여부를 확인합니다.
if torch.cuda.is_available():  # CUDA가 활성화된 환경에서 GPU 이름을 표시합니다.
    print('GPU:', torch.cuda.get_device_name(0))  # RTX 3070 Ti가 보이는지 확인합니다.

## 데이터 분할

공식 정상 reference 552개 lot을 fit 441개와 validation/calibration 111개로 나눈다. 공개 holdout은 정상 237개와 이상 367개다. 같은 lot의 시계열이 서로 다른 split에 섞이지 않도록 ID 단위로 분리한다.

정상 111개는 epoch와 threshold 선택에 함께 사용한다. 같은 holdout에서 여러 모델과 점수를 비교했으므로 결과는 탐색 benchmark로 해석한다.


In [ ]:
import st_awfd_causal_gpt_multihorizon as experiment  # 재현 스크립트의 모델·split·학습 함수를 가져옵니다.
train, calibration, holdout, holdout_labels, train_ids, calibration_ids, holdout_ids, scaler = experiment.prepare_data()  # 공식 group split과 training-only scaler를 불러옵니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # CUDA가 있으면 RTX GPU를 우선 사용합니다.
model = experiment.CausalMultiHorizonGPT(feature_count=train.shape[1], sequence_length=train.shape[-1]).to(device)  # 20개 sensor와 64-point trace에 맞는 모델을 만듭니다.
parameter_count = sum(parameter.numel() for parameter in model.parameters())  # 실제 학습 대상 parameter 수를 계산합니다.
print('Train normals:', len(train_ids), 'shape:', train.shape)  # fitting 표본 수와 tensor shape를 확인합니다.
print('Calibration normals:', len(calibration_ids), 'shape:', calibration.shape)  # threshold reference 표본 수를 확인합니다.
print('Official held-out groups:', len(holdout_ids), 'shape:', holdout.shape, 'abnormal:', int(holdout_labels.sum()))  # 최종 사후 평가 표본과 label count를 봅니다.
print('Architecture parameters:', f'{parameter_count:,}')  # 소형 모델의 크기를 기록합니다.
print('Forecast horizons:', experiment.HORIZONS, 'device:', device)  # 각 token이 예측하는 future lead와 장치를 확인합니다.

## Causal 경로

왼쪽 padding Conv1d와 attention mask가 시점 t 이후의 입력을 차단한다. cutoff 뒤 센서값을 바꾸고 cutoff 시점의 예측 평균과 log-sigma가 유지되는지 검사한다.

검사 대상은 보간된 tensor 안의 신경망이다. 전처리는 전체 step의 종료점과 선형 보간을 사용하므로 완료 trace의 사후 분석에 해당한다.


In [ ]:
dummy_input = torch.from_numpy(train[:1, :, :-1]).to(device)  # 예시 정상 trace 한 개에서 마지막 관측값을 목표로 남깁니다.
with torch.no_grad():  # shape 확인에는 gradient가 필요하지 않습니다.
    forecast_mean, forecast_log_sigma = model(dummy_input)  # 현재 모델의 4 horizon별 Gaussian forecast를 계산합니다.
print('Input shape [batch,sensor,time]:', tuple(dummy_input.shape))  # sensor/time tensor 순서를 확인합니다.
print('Output mean shape [batch,time,horizon,sensor]:', tuple(forecast_mean.shape))  # 각 과거 token과 horizon의 forecast를 확인합니다.
causal_check = experiment.causal_invariance_audit(model, train[0], device)  # 학습 전 구조에서 future perturbation 불변성도 계산합니다.
print(causal_check)  # maximum output difference가 numerical zero인지 확인합니다.

## 정상 분포 학습

Gaussian NLL로 예측 평균과 표준편차를 함께 학습한다. 연속 확률밀도는 1보다 클 수 있어 NLL이 음수가 될 수 있다. CNN MSE와 loss의 숫자 자체를 비교하지 않고 별도 탐지·오경보 지표를 계산한다.

최대 320 epoch와 patience 30을 사용한다. 정상 validation NLL이 가장 낮은 weight를 복원하고 정상 점수의 95th percentile을 경보 기준으로 고정한다.


In [ ]:
from IPython.display import Image, display  # 저장된 actual run chart를 노트북 안에 표시합니다.
import pandas as pd  # metric과 epoch table을 읽기 쉽게 렌더링합니다.
epoch_events = []  # callback으로 받은 실제 epoch 수치만 누적합니다.
def show_epoch(event):  # 완료된 optimizer epoch의 값을 notebook output에 보냅니다.
    epoch_events.append({'epoch': event['epoch'], 'train_nll': event['train_loss'], 'normal_validation_nll': event['validation_loss']})  # train/validation loss를 epoch 단위로 기록합니다.
    if event['epoch'] == 1 or event['epoch'] % 10 == 0 or event['epoch'] == event['max_epochs']:  # 화면이 지나치게 길어지지 않도록 10회 간격을 표시합니다.
        print(f"epoch {event['epoch']:03d}/{event['max_epochs']} | train {event['train_loss']:.4f} | normal-val {event['validation_loss']:.4f} | best {event['best_validation_loss']:.4f} @ {event['best_epoch']} | {event['device']}")  # 실제 모델 loss와 학습 장치를 출력합니다.
result = experiment.run_experiment(progress_callback=show_epoch)  # 3070 Ti 학습, early stopping, calibration, inference, Excel/CSV를 실행합니다.
display(pd.DataFrame([result['metrics']]).T.rename(columns={0: 'measured value'}))  # 같은 run에서 나온 다중 지표를 세로 표로 표시합니다.
print('Selected epoch:', result['best_epoch'], '| Device:', result['device'])  # 이번 execution이 실제 선택한 epoch와 device를 확인합니다.
print('Causality audit:', result['summary']['causality_audit'])  # 학습된 모델에서 미래 정보 누출이 없는지 측정 결과를 표시합니다.
display(Image(filename=str(result['chart_path'])))  # loss와 official holdout score가 저장된 PNG를 노트북에 보입니다.

## 모델 비교

CNN과 Transformer는 같은 D2 split과 정상 reference tail을 사용한다. 순위 지표, 고정 기준의 recall·precision·오경보, 검토 용량을 함께 비교한다.


In [ ]:
comparison = pd.read_csv(result['comparison_path'])  # CNN, strict-causal Transformer 및 고전 baseline의 비교 결과를 읽습니다.
display(comparison)  # AP/AUC만 보지 않고 FPR, precision, recall, review capture를 나란히 봅니다.
audit_path = project_dir / 'results' / 'st_awfd' / 'causal_gpt_multihorizon' / 'official_holdout_audit_scores.csv'  # label 포함 test audit file 경로입니다.
audit = pd.read_csv(audit_path)  # sensor별·horizon별 contribution과 retrospective labels를 불러옵니다.
feature_columns = [column for column in audit.columns if column.startswith('feature_') and column.endswith('_mean_nll')]  # 익명 sensor score 열만 골라냅니다.
horizon_columns = [column for column in audit.columns if column.startswith('horizon_') and column.endswith('_mean_nll')]  # 네 forecast 거리별 score 열을 고릅니다.
audit['leading_sensor_for_review'] = audit[feature_columns].idxmax(axis=1)  # 각 group의 상대적으로 가장 높은 sensor surprise를 정리합니다.
audit['leading_horizon_for_review'] = audit[horizon_columns].idxmax(axis=1)  # 짧은/긴 거리 중 가장 큰 score horizon을 찾습니다.
display(audit.sort_values('multi_horizon_gaussian_nll', ascending=False).head(20)[['MaterialID', 'target_for_retrospective_audit_only', 'review_rank', 'multi_horizon_gaussian_nll', 'leading_sensor_for_review', 'leading_horizon_for_review']])  # top anomaly audit rows를 보여줍니다.
print('This table is a retrospective benchmark audit; feature_ names are anonymous and are not physical root-cause labels.')  # sensor 번호를 fab physical cause로 오해하지 않게 합니다.

## 잔차 공분산 점수

고정된 Transformer의 정상 학습 잔차에 horizon별 Ledoit–Wolf 공분산을 맞추고, Mahalanobis energy를 시간·horizon 평균으로 집계한다. Gaussian NLL과 달리 log-sigma 항을 쓰지 않는 별도 점수다.

기록된 비교에서는 정상 오경보가 76개에서 70개로 줄었지만 AP와 상위 검토 큐의 성능은 낮아졌다. 아래 cell은 별도 공분산 실행에서 저장한 결과를 읽는다.


In [ ]:
import json  # 실제 저장된 covariance 탐색 manifest를 읽습니다.
if (covariance_dir / 'covariance_exploratory_summary.json').is_file():
    import pandas as pd  # 여러 지표를 표로 표시합니다.
    from pathlib import Path  # 현재 프로젝트 기준의 결과 파일을 지정합니다.
    from IPython.display import display  # 학습 셀을 생략해도 표를 볼 수 있게 합니다.
    covariance_dir = Path('results/st_awfd/gpt_covariance_exploratory')  # 기존 GPT 결과와 구분된 추가 탐색 결과 폴더입니다.
    covariance_summary = json.loads((covariance_dir / 'covariance_exploratory_summary.json').read_text(encoding='utf-8'))  # 실제 실행된 fit scope, weight hash와 지표를 가져옵니다.
    print('New neural epochs:', covariance_summary['new_neural_training_epochs'], '| checkpoint unchanged:', covariance_summary['checkpoint_unchanged'])  # 점수 후처리만 바꿨음을 확인합니다.
    display(pd.DataFrame([covariance_summary['metrics']]).T.rename(columns={0: 'exploratory measured value'}))  # FPR 개선과 ranking/queue 손실을 함께 검토합니다.
    display(pd.read_csv(covariance_dir / 'model_comparison_exploratory.csv'))  # 기존 GPT NLL과 공분산 score의 비교표를 표시합니다.
    print('Repeatedly inspected official holdout: exploratory evidence, not a fresh confirmatory final test.')  # 연구 선택에 재사용된 데이터임을 출력에도 명시합니다.

else:
    print('저장된 공분산 비교 결과 없음')


## 저장 모델 CSV 추론

저장된 weight, train scaler, 경보 기준으로 새 CSV를 점수화한다. 입력 열은 MaterialID, StepID, duration_ms, feature_1…feature_20이다.

above_normal_reference_limit은 고정 threshold 초과 여부이고, review_candidate_at_capacity는 batch의 상위 검토 용량이다. 익명 feature와 예측 horizon 기여도는 점수의 계산 근거로 표시한다.


In [ ]:
from pathlib import Path  # 프로젝트 폴더를 기준으로 demo 입력과 결과 경로를 지정합니다.
from st_awfd_causal_gpt_multihorizon import score_new_csv  # 학습 완료 checkpoint를 재사용하는 inference API를 가져옵니다.
from IPython.display import display  # 재학습 셀을 건너뛰어도 표를 노트북에 표시할 수 있게 합니다.
input_csv = Path('results/st_awfd/st_awfd_label_free_demo_input.csv')  # label을 제거한 공식 holdout 시연 파일을 사용합니다.
if not input_csv.exists():  # 입력 파일이 없는 상태로 잘못된 추론을 시작하지 않습니다.
    raise FileNotFoundError(f'Missing demo CSV: {input_csv.resolve()}')  # 필요한 파일 경로를 구체적으로 알립니다.
inference_queue = score_new_csv(input_csv, review_fraction=0.10)  # 고정 모델/scaler/threshold로만 새로운 batch를 점수화합니다.
review_columns = ['MaterialID', 'anomaly_score', 'normal_reference_limit', 'above_normal_reference_limit', 'review_rank', 'review_candidate_at_capacity', 'leading_anonymous_feature', 'largest_surprise_horizon']  # score, limit, 순위와 PE 검토 후보 정보를 함께 봅니다.
display(inference_queue[review_columns].head(20))  # 우선 검토할 group을 score 순서로 보여줍니다.
output_csv = Path('results/st_awfd/causal_gpt_multihorizon/new_batch_review_queue.csv')  # 재학습 결과와 구분할 신규 batch queue 경로입니다.
output_csv.parent.mkdir(parents=True, exist_ok=True)  # CSV를 쓸 결과 폴더가 없으면 준비합니다.
inference_queue.to_csv(output_csv, index=False, encoding='utf-8-sig')  # Excel에서 한글과 열 이름을 읽기 쉽게 UTF-8 BOM으로 저장합니다.
print('Saved:', output_csv.resolve())  # 실제로 저장한 결과 파일을 확인합니다.
print('Inference used saved weights, train scaler, and fixed threshold; no fitting was performed.')  # 새 학습 없이 추론만 실행했다는 범위를 명시합니다.
